# TFG V2 — Generalization to multidimensional grids

This version extends the 1D prototype to grids of arbitrary dimension. `N` defines the grid size along each dimension, and `M` defines the window or job size along each dimension.

## What the circuit does

1. Computes every valid starting coordinate for a window `M` inside a grid `N`.
2. Encodes each starting coordinate as a linear index `idx` using row-major order.
3. Reversibly loads the multidimensional window `window_i` into the register `m`.

The target state remains:

$\sum_i |grid>|idx=i>|window_i>$

but now `i` represents a starting coordinate in an ND grid.

## Key functions

- `coord_to_index(...)` and `index_to_coord(...)`: conversion between ND coordinates and linear indices.
- `valid_starts_nd(...)`: generates all valid starting positions.
- `window_qubits_nd(...)`: returns the cells covered by a window.

## Purpose of this version

V2 turns the project into a genuinely multidimensional problem. It still does not attempt to amplify solutions; its role is to ensure that the ND geometry, index encoding, and window loading are coherent.

In [ ]:
import time
import qiskit
import numpy as np

from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister
from qiskit.quantum_info import Statevector, SparsePauliOp, partial_trace
from qiskit.visualization import plot_bloch_multivector, plot_state_qsphere
from qiskit_aer import AerSimulator
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager
# DISABLED: IBM Runtime import; using simulators only. from qiskit_ibm_runtime import EstimatorV2 as Estimator
# DISABLED: IBM Runtime import; using simulators only. from qiskit_ibm_runtime import SamplerV2 as Sampler
# DISABLED: IBM Runtime import; using simulators only. from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit.visualization import plot_histogram
from itertools import product
from math import prod

print(qiskit.__version__)

In [ ]:
# =========================================================
# GENERAL HELPER FUNCTIONS FOR D DIMENSIONS
# =========================================================

def index_to_coord(index, dims):
    """
    Converts a row-major linear index to a D-dimensional coordinate.
    """
    coord = [0] * len(dims)
    rem = index
    for d in reversed(range(len(dims))):
        coord[d] = rem % dims[d]
        rem //= dims[d]
    return tuple(coord)

def coord_to_index(coord, dims):
    """
    Converts a D-dimensional coordinate to a row-major linear index.
    """
    idx_lin = 0
    stride = 1
    for d in reversed(range(len(dims))):
        idx_lin += coord[d] * stride
        stride *= dims[d]
    return idx_lin

def reshape_bits_nd(bitstring, dims):
    """
    Reorganiza un bitstring lineal row-major a una estructura anidada
    con forma dims.
    """
    arr = np.array(list(bitstring), dtype=str)
    return arr.reshape(dims)

def format_nd_array(arr, indent=0):
    """
    Formats an ndarray/string-array of any dimension in a readable way.
    """
    if arr.ndim == 1:
        return "[" + " ".join(arr.tolist()) + "]"

    inner = []
    for sub in arr:
        inner.append(format_nd_array(np.array(sub), indent + 2))

    sep = ",\n" + " " * indent
    return "[\n" + " " * indent + sep.join(inner) + "\n" + " " * max(indent - 2, 0) + "]"

def valid_starts_nd(N, M):
    """
    List of valid starting coordinates for the window in D dimensions.
    Usa la misma convencion row-major.
    """
    shape_starts = tuple(N[d] - M[d] + 1 for d in range(len(N)))
    return list(np.ndindex(shape_starts))

def window_qubits_nd(start, N, M):
    """
    Returns the linear qubit indices of the window that starts at 'start',
    en orden row-major dentro del bloque M.

    2D example with M=[2,2]:
    offsets visitados:
        (0,0), (0,1), (1,0), (1,1)
    """
    qubits = []
    for offset in np.ndindex(tuple(M)):
        coord = tuple(start[d] + offset[d] for d in range(len(N)))
        qubits.append(coord_to_index(coord, N))
    return qubits

In [ ]:
# =========================
# Parameters
# =========================

D = 2
N = [4, 4]   # grid size in each dimension
M = [2, 2]   # job/window size in each dimension

if len(N) != D or len(M) != D:
    raise ValueError("N y M deben tener longitud D")

for d in range(D):
    if M[d] > N[d]:
        raise ValueError(f"In dimension {d}, M[{d}] cannot be greater than N[{d}]")

# Total number of cells (grid and work register) and valid windows in D dimensions
N_tot = prod(N)
M_tot = prod(M)
starts = valid_starts_nd(N, M)
W = len(starts)

# Required index qubits
k = int(np.ceil(np.log2(W))) if W > 1 else 1

print(f"Dimensions: {D}")
print(f"Grid size per dimension: {N}")
print(f"Job/window size per dimension: {M}")
print(f"Total cells: {N_tot}")
print(f"Total job/window cells: {M_tot}")
print(f"Total possible windows: {W}")
print(f"Qubits required to represent the windows: {k}")

print("\nPossible valid windows (start coordinate):")
for idx_start, start in enumerate(starts):
    print(f"{idx_start}: {start} -> {window_qubits_nd(start, N, M)}")

# =========================
# Registers
# =========================

n = QuantumRegister(N_tot, "n")   # red
idx = QuantumRegister(k, "i")     # window index
m = QuantumRegister(M_tot, "m")   # work window

qc = QuantumCircuit(n, idx, m)

# =========================
# Example initial state
# =========================

# 2x2 square at (0,0) in N=4x4
# qc.x(n[0])
# qc.x(n[1])
# qc.x(n[4])
# qc.x(n[5])

# 2x2 square at (2,1) in N=4x4
# qc.x(n[9])
# qc.x(n[10])
# qc.x(n[13])
# qc.x(n[14])

# # Corners in N=3x3
# qc.x(n[0])
# qc.x(n[2])
# qc.x(n[6])
# qc.x(n[8])

# Corners in N=4x4
qc.x(n[0])
qc.x(n[3])
qc.x(n[12])
qc.x(n[15])

# =========================
# Superposition over ONLY valid indices
# =========================
# Prepares:
#   (1/sqrt(W)) * sum_{i=0}^{W-1} |i>
# and leaves amplitude 0 on invalid indices i >= W

amps_idx = np.zeros(2**k, dtype=complex)
amps_idx[:W] = 1 / np.sqrt(W)
qc.initialize(amps_idx, idx)

# =========================
# Gray-like order to minimize X gates on idx
# =========================
# We traverse valid indices in an order close to Gray:
# take the full Gray sequence over k bits and filter valid values < W

gray_full = [t ^ (t >> 1) for t in range(2**k)]
order_valid = [g for g in gray_full if g < W]

print("\nGray-like order used to load windows:")
print(order_valid)

# =========================
# Coherent window loading
# =========================
# Instead of:
#   activating X according to i
#   applying MCX
#   undoing X
# for each i,
# we keep the current X pattern on idx and only change
# the bits that actually vary between one window and the next.

current_zero_mask = [False] * k  # False = no hay X puesta en idx[b]

for i in order_valid:
    bits = [(i >> b) & 1 for b in range(k)]   # little-endian
    target_zero_mask = [bits[b] == 0 for b in range(k)]
    win = window_qubits_nd(starts[i], N, M)

    # Only toggle idx[b] qubits whose X/no-X state changes
    for b in range(k):
        if current_zero_mask[b] != target_zero_mask[b]:
            qc.x(idx[b])
            current_zero_mask[b] = target_zero_mask[b]

    # Copy the window onto m with XOR
    for j, n_pos in enumerate(win):
        controls = [idx[b] for b in range(k)] + [n[n_pos]]
        qc.mcx(controls, m[j])

# Clear the final X pattern on idx
for b in range(k):
    if current_zero_mask[b]:
        qc.x(idx[b])

qc.draw(output='mpl')

In [ ]:
sv = Statevector(qc)
sv.draw(output='latex', max_size=2**qc.num_qubits, prefix="|\\psi\\rangle = ")

In [ ]:
# =========================================================
# 1) Remove the n register
# =========================================================
rho = partial_trace(sv, list(range(N_tot)))

# =========================================================
# 2) Check purity
# =========================================================
purity = np.real(np.trace(rho.data @ rho.data))
print(f"Purity = {purity}")

if not np.isclose(purity, 1.0, atol=1e-10):
    raise ValueError("El estado reducido no es puro; no se puede representar como un unico ket.")

# =========================================================
# 3) Extract the complete n-position array
# =========================================================
tol_n = 1e-10
total_q = qc.num_qubits
array_posiciones = None

for basis_idx, amp in enumerate(sv.data):
    if abs(amp) <= tol_n:
        continue

    bits_full = format(basis_idx, f'0{total_q}b')

    # In Qiskit, the string is returned as:
    # m_{M_tot-1}...m_0  i_{k-1}...i_0  n_{N_tot-1}...n_0
    # Therefore the n qubits are at the end.
    n_desc = bits_full[-N_tot:]    # n_{N_tot-1}...n_0
    n_asc = n_desc[::-1]           # n_0...n_{N_tot-1}

    if array_posiciones is None:
        array_posiciones = n_asc
    elif n_asc != array_posiciones:
        raise ValueError("The n qubits are not deterministic (there is no unique position array).")

if array_posiciones is None:
    raise ValueError("Could not infer the position array from the statevector.")

print(f"Linear position array = {array_posiciones}\n")

# Also display the grid with shape N
red_nd = reshape_bits_nd(array_posiciones, N)
print(f"Red con forma {tuple(N)}:")
print(format_nd_array(red_nd))
print()

# =========================================================
# 4) Extract the ket of the reduced state
# =========================================================
vals, vecs = np.linalg.eigh(rho.data)
psi_red = Statevector(vecs[:, np.argmax(vals)])

# =========================================================
# 5) Extract labels from the reduced state
# =========================================================
# In psi_red, the binary string has the form:
# m_{M_tot-1}...m_0  i_{k-1}...i_0
total_bits = M_tot + k
tol = 1e-10
filas = []

for basis_idx, amp in enumerate(psi_red.data):
    if abs(amp) <= tol:
        continue

    bits = format(basis_idx, f'0{total_bits}b')

    m_desc = bits[:M_tot]     # m_{M_tot-1}...m_0
    i_desc = bits[M_tot:]     # i_{k-1}...i_0

    ventana_lineal = m_desc[::-1]   # m_0...m_{M_tot-1}
    indices = i_desc
    idx_int = int(indices, 2)

    filas.append((idx_int, indices, ventana_lineal))

# Sort by the integer value of the index
filas.sort(key=lambda x: (x[0], x[2]))

# Separate valid and invalid indices
filas_validas = [fila for fila in filas if fila[0] < W]
filas_invalidas = [fila for fila in filas if fila[0] >= W]

# =========================================================
# 6) Display valid windows with ND shape
# =========================================================
print("Valid windows:\n")

for idx_int, indices, ventana_lineal in filas_validas:
    start = starts[idx_int]   # ND start coordinate corresponding to this index
    ventana_nd = reshape_bits_nd(ventana_lineal, M)

    print(f"Index: {indices} ({idx_int})")
    print(f"Start:  {start}")
    print(f"Ventana lineal: {ventana_lineal}")
    print(f"Ventana con forma {tuple(M)}:")
    print(format_nd_array(ventana_nd))
    print()

# =========================================================
# 7) Display invalid indices, if any
# =========================================================
if filas_invalidas:
    print("----------------------------------------")
    print("Indices outside the valid window range:\n")

    for idx_int, indices, ventana_lineal in filas_invalidas:
        ventana_nd = reshape_bits_nd(ventana_lineal, M)

        print(f"Index: {indices} ({idx_int})")
        print(f"Ventana lineal: {ventana_lineal}")
        print(f"Ventana con forma {tuple(M)}:")
        print(format_nd_array(ventana_nd))
        print()